# Create Production and Heavy Production Monthly Gold Tables

## Purpose
Data promotion notebook that creates gold-layer production time series tables from silver-layer cleaned EIA data. This notebook prepares production data for downstream forecasting and analysis.

## Data Flow

**Data Sources (Read):**
* `workspace.silver.eia_countrylvl_heavy_prod_calculated` - Cleaned heavy oil production data with API calculations
* `workspace.silver.eia_oilcond_production_monthly` - Cleaned monthly total crude production data
* `workspace.silver.eia_oilcond_production_annual` - Cleaned annual total crude production data

**Data Destinations (Write):**
* `workspace.gold.heavy_prod_timeseries` - Annual heavy oil production time series (Country, Year, Heavy_Oil_Prod)
* `workspace.gold.prod_timeseries_monthly` - Monthly total crude production time series (Country, Date, value)
* `workspace.gold.prod_timeseries_annual` - Annual total crude production time series (Country, Year, value)

## Workflow
1. **Heavy production annual** - Extract heavy oil production by country and year → Write to gold.heavy_prod_timeseries
2. **Total production monthly** - Extract monthly crude production → Write to gold.prod_timeseries_monthly
3. **Total production annual** - Extract annual crude production → Write to gold.prod_timeseries_annual

## Key Features
* **Full refresh** - All tables use overwrite mode with schema overwrite enabled
* **Schema promotion** - Simplifies silver schemas by selecting only essential columns for gold layer
* **Time series ready** - Outputs are structured for direct use in forecasting models
* **Validation steps** - Includes row count checks and data preview

In [0]:

df = spark.sql("SELECT Country, Year, Heavy_Oil_Prod FROM workspace.silver.eia_countrylvl_heavy_prod_calculated")



In [0]:
# Write as a Delta table in Unity Catalog (UC)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.heavy_prod_timeseries")

In [0]:
df = spark.sql("SELECT * FROM workspace.silver.eia_oilcond_production_monthly")

#print the total number of rows
print(df.count())

df.show(10)

47875
+-------------------+-------+--------------------+--------------------+---------+
|            Country|   Date|         productName|            unitName|    value|
+-------------------+-------+--------------------+--------------------+---------+
|               Chad|2025-06|Crude oil includi...|thousand barrels ...|    127.0|
|           Thailand|2025-06|Crude oil includi...|thousand barrels ...|    158.0|
|         Tajikistan|2025-06|Crude oil includi...|thousand barrels ...|    0.316|
|       Turkmenistan|2025-06|Crude oil includi...|thousand barrels ...|190.65793|
|       Timor-Leste |2025-06|Crude oil includi...|thousand barrels ...|      2.0|
|Trinidad and Tobago|2025-06|Crude oil includi...|thousand barrels ...|49.638515|
|            Tunisia|2025-06|Crude oil includi...|thousand barrels ...|     26.0|
|            Turkiye|2025-06|Crude oil includi...|thousand barrels ...|  123.609|
|             Taiwan|2025-06|Crude oil includi...|thousand barrels ...|    0.196|
|      Uni

In [0]:
df = spark.sql("SELECT Country, Date, value FROM workspace.silver.eia_oilcond_production_monthly")

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.prod_timeseries_monthly")


In [0]:
df = spark.sql("SELECT * FROM workspace.silver.eia_oilcond_production_annual")

#print the total number of rows
print(df.count())

df.show(10)

3729
+--------------------+----+--------------------+--------------------+---------+
|             Country|Year|         productName|            unitName|    value|
+--------------------+----+--------------------+--------------------+---------+
|Former Serbia and...|1992|Crude oil includi...|thousand barrels ...|     23.0|
|            Suriname|1992|Crude oil includi...|thousand barrels ...|      4.0|
|               Syria|1992|Crude oil includi...|thousand barrels ...|  520.082|
|            Thailand|1992|Crude oil includi...|thousand barrels ...|52.084698|
|          Tajikistan|1992|Crude oil includi...|thousand barrels ...|      1.0|
|        Turkmenistan|1992|Crude oil includi...|thousand barrels ...|     85.0|
| Trinidad and Tobago|1992|Crude oil includi...|thousand barrels ...|136.23224|
|             Tunisia|1992|Crude oil includi...|thousand barrels ...|    114.0|
|              Taiwan|1992|Crude oil includi...|thousand barrels ...|      2.0|
|             Ukraine|1992|Crude oi

Create the annual data

In [0]:
df = spark.sql("SELECT Country, Year, value FROM workspace.silver.eia_countrylvl_heavy_prod_calculated")

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.prod_timeseries_annual")